In [6]:
import cdsapi
import xarray as xr
import zipfile
from pathlib import Path

In [2]:
# do my credentials work?
client = cdsapi.Client()

In [4]:
import cdsapi

test = "satellite-soil-moisture"
request = {
    "variable": ["surface_soil_moisture_volumetric"],
    "type_of_sensor": ["passive"],
    "time_aggregation": ["daily"],
    "year": [
        "2014"
    ],
    "month": [
        "01"
    ],
    "day": [
        "01"
    ],
    "type_of_record": ["cdr"],
    "version": ["v202505"]
}

client = cdsapi.Client()
result =client.retrieve(test, request)


2026-10-02 12:11:00,094 INFO Request ID is 5f75ca59-4e2d-450c-b178-fc18b389a99e
2026-10-02 12:11:00,623 INFO status has been updated to successful


In [5]:
print(result)
print(type(result))

downloaded = result.download()
print(downloaded)
print(type(downloaded))

Results(response=<Response [200]>, headers={'User-Agent': 'ecmwf-datastores-client/0.5.3', 'PRIVATE-TOKEN': '9a013d63-db93-4cdf-bac1-73c22a2ef971'}, session=<requests.sessions.Session object at 0x0000024F27DD3F50>, retry_options={'maximum_tries': 500, 'retry_after': 120}, request_options={'timeout': 60, 'verify': True}, download_options={'progress_bar': <function progress_bar at 0x0000024F280A3380>}, sleep_max=120, cleanup=False, log_callback=<bound method LegacyClient.log of <ecmwf.datastores.legacy_client.LegacyClient object at 0x0000024F275C74A0>>)
<class 'ecmwf.datastores.processing.Results'>


33a11b82dbc4691cba71d83cc57b8a10.zip
<class 'str'>


In [10]:
zip_path = Path(downloaded)
with zipfile.ZipFile(zip_path, "r") as z:
    print(z.namelist())
    nc_file = [name for name in z.namelist() if name.endswith(".nc")][0]
    z.extract(nc_file, path=zip_path.parent)

nc_path = zip_path.parent / nc_file

ds = xr.open_dataset(nc_path, engine="netcdf4")

print(ds)
print(ds.coords)
print(ds.data_vars)
print(ds.attrs)

['C3S-SOILMOISTURE-L3S-SSMV-PASSIVE-DAILY-20140101000000-TCDR-v202505.0.0.nc']
<xarray.Dataset> Size: 29MB
Dimensions:         (time: 1, lat: 720, lon: 1440)
Coordinates:
  * time            (time) datetime64[ns] 8B 2014-01-01
  * lat             (lat) float32 3kB 89.88 89.62 89.38 ... -89.38 -89.62 -89.88
  * lon             (lon) float32 6kB -179.9 -179.6 -179.4 ... 179.4 179.6 179.9
Data variables:
    sm              (time, lat, lon) float32 4MB ...
    sm_uncertainty  (time, lat, lon) float32 4MB ...
    flag            (time, lat, lon) float32 4MB ...
    freqbandID      (time, lat, lon) int16 2MB ...
    dnflag          (time, lat, lon) int8 1MB ...
    mode            (time, lat, lon) int8 1MB ...
    sensor          (time, lat, lon) int32 4MB ...
    t0              (time, lat, lon) datetime64[ns] 8MB ...
Attributes: (12/44)
    title:                        C3S Satellite Surface Soil Moisture merged ...
    institution:                  TU Wien (AUT); Planet Labs (NL); EODC G

lat is descending and lon ascending??

In [11]:
print(ds["sm"])
print(ds["sm"].attrs)

<xarray.DataArray 'sm' (time: 1, lat: 720, lon: 1440)> Size: 4MB
[1036800 values with dtype=float32]
Coordinates:
  * time     (time) datetime64[ns] 8B 2014-01-01
  * lat      (lat) float32 3kB 89.88 89.62 89.38 89.12 ... -89.38 -89.62 -89.88
  * lon      (lon) float32 6kB -179.9 -179.6 -179.4 -179.1 ... 179.4 179.6 179.9
Attributes:
    dtype:                float32
    units:                m3 m-3
    valid_range:          [0. 1.]
    long_name:            Volumetric Soil Moisture
    ancillary_variables:  sm_uncertainty flag t0
{'dtype': 'float32', 'units': 'm3 m-3', 'valid_range': array([0., 1.], dtype=float32), 'long_name': 'Volumetric Soil Moisture', 'ancillary_variables': 'sm_uncertainty flag t0'}


In [1]:
from importfuncs import _search_copernicus_sm

products_2014 = _search_copernicus_sm(2014)
products_2015 = _search_copernicus_sm(2015)
products_2016 = _search_copernicus_sm(2016)
products_2017 = _search_copernicus_sm(2017)
products_2018 = _search_copernicus_sm(2018)
products_2019 = _search_copernicus_sm(2019)
products_2020 = _search_copernicus_sm(2020)
products_2021 = _search_copernicus_sm(2021)
products_2022 = _search_copernicus_sm(2022)
products_2023 = _search_copernicus_sm(2023)
products_2024 = _search_copernicus_sm(2024)
products_2025 = _search_copernicus_sm(2025)

print(len(products_2014))
print(products_2014[0]["Name"])
print(products_2014[-1]["Name"])

178
c_gls_SSM1km_201410030000_CEURO_S1CSAR_V1.1.1_cog
c_gls_SSM1km_201412310000_CEURO_S1CSAR_V1.1.1_cog


In [2]:
nc_products_2014 = [
    product
    for product in products_2014
    if product["Name"].endswith("_nc")
]

In [ ]:
len(nc_products_2014)

In [ ]:
dates_2014 = [
    product["ContentDate"]["Start"][:10]
    for product in nc_products_2014
]

len(dates_2014), len(set(dates_2014))

In [ ]:
print(dates_2014[:5])
print(dates_2014[-5:])

In [5]:
for year, products in [
    (2014, products_2014),
    (2015, products_2015),
    (2016, products_2016),
    (2017, products_2017),
    (2018, products_2018),
    (2019, products_2019),
    (2020, products_2020),
    (2021, products_2021),
    (2022, products_2022),
    (2023, products_2023),
    (2024, products_2024),
    (2025, products_2025)
]:
    nc_products = [
        p for p in products
        if p["Name"].endswith("_nc")
    ]

    dates = [
        p["ContentDate"]["Start"][:10]
        for p in nc_products
    ]

    print(year)
    print("  number of products:", len(nc_products))
    print("  first:", dates[0])
    print("  last: ", dates[-1])

2014
  number of products: 89
  first: 2014-10-03
  last:  2014-12-31
2015
  number of products: 360
  first: 2015-01-01
  last:  2015-12-31
2016
  number of products: 356
  first: 2016-01-01
  last:  2016-12-31
2017
  number of products: 365
  first: 2017-01-01
  last:  2017-12-31
2018
  number of products: 333
  first: 2018-01-01
  last:  2018-12-31
2019
  number of products: 365
  first: 2019-01-01
  last:  2019-12-31
2020
  number of products: 365
  first: 2020-01-01
  last:  2020-12-31
2021
  number of products: 364
  first: 2021-01-01
  last:  2021-12-31
2022
  number of products: 364
  first: 2022-01-01
  last:  2022-12-31
2023
  number of products: 365
  first: 2023-01-01
  last:  2023-12-31
2024
  number of products: 366
  first: 2024-01-01
  last:  2024-12-31
2025
  number of products: 365
  first: 2025-01-01
  last:  2025-12-31
